In [1]:
!uv pip list

Package                                  Version     Editable project location
---------------------------------------- ----------- ----------------------------
aiohappyeyeballs                         2.6.1
aiohttp                                  3.12.13
aiohttp-retry                            2.9.1
aiosignal                                1.3.2
annotated-types                          0.7.0
anthropic                                0.54.0
anyio                                    4.9.0
asttokens                                3.0.0
asyncstdlib-fw                           3.13.2
attrs                                    25.3.0
backoff                                  2.2.1
bcrypt                                   4.3.0
betterproto-fw                           2.0.3
blockbuster                              1.5.24
build                                    1.2.2.post1
cachetools                               5.5.2
certifi                                  2025.6.15
cffi                    

Using Python 3.12.11 environment at: C:\cursor\langgraph_baseline\.venv


# RAG 시스템 설계
하나하나 기능 단위로 시작

In [3]:
from dotenv import load_dotenv
import os
import json


load_dotenv()
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

In [3]:
from pathlib import Path

# 현재 파일의 경로 객체 생성
current_file_path = Path.cwd()
print(f"현재 파일의 경로 객체: {current_file_path}")

# 현재 파일이 있는 디렉토리 경로
current_dir = current_file_path.parent
print(f"현재 파일의 디렉토리: {current_dir}")

# 데이터 파일 동적 경로 생성 : "/"  연산자 사용
file_path = current_dir / 'final_merged_products.jsonl'
print(f"데이터 파일의 동적 경로: {file_path}")

현재 파일의 경로 객체: c:\cursor\langgraph_baseline\oliveyoung
현재 파일의 디렉토리: c:\cursor\langgraph_baseline
데이터 파일의 동적 경로: c:\cursor\langgraph_baseline\final_merged_products.jsonl


### 0. JSONL 파악

In [4]:
import json

# 집 경로
# file_path = 'C:\\cursor\\DealMakers\\langgraph_baseline\\oliveyoung\\final_merged_products.jsonl'

# 회사 경로
# file_path = "C:\\cursor\\langgraph_baseline\\oliveyoung\\final_merged_products.jsonl"
file_path = current_file_path / 'final_merged_products.jsonl'

count = 0
max_count = 2 # 확인하고 싶은 객체 수

with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        if count >= max_count:
            break # max_count 만큼 확인했으면 반복문 종료

        try:
            obj = json.loads(line)
            print(f"--- 객체 {count + 1} ---")
            print(obj)
            count += 1
        except json.JSONDecodeError:
            print(f"다음 줄에서 JSON 파싱 오류 발생: {line.strip()}")

--- 객체 1 ---
{'product_name': '[7.12 하루특가/7월 올영픽]라로슈포제 시카플라스트 멀티 리페어 크림 B5 100ml 기획 (멜라B3 세럼 3ml)', 'brand': '라로슈포제', 'category_path': ['스킨케어', '크림', '크림'], 'original_price': 50000, '용량': '100ml+3ml', '제조/판매업자': '프랑스 라로슈포제사 / 엘오케이 유한회사', '전성분': '시카플라스트 멀티 리페어 크림\n정제수 다이카프릴릴에터 판테놀 글리세린 펜틸렌글라이콜 폴리글리세릴-6다이스테아레이트 프로판다이올 세틸에스터 호호바에스터 베헤닐알코올 그린와틀꽃왁스 해바라기씨왁스 병풀잎추출물 야콘뿌리즙 솔비탄올리에이트 징크글루코네이트 마데카소사이드 세테아릴아이소노나노에이트 망가니즈글루코네이트 아이소헥사데칸 알파-글루칸올리고사카라이드 소듐하이알루로네이트 소듐스테아로일글루타메이트 아데노신 만노오스 카퍼글루코네이트 하이드록시아세토페논 하이드록시프로필스타치포스페이트 카프릴로일살리실릭애씨드 비트레오스실라발효물 시트릭애씨드 트라이소듐에틸렌다이아민다이석시네이트 락토바실러스 말토덱스트린 폴리글리세린-3 폴리글리세릴-3비즈왁스 폴리솔베이트80 아크릴아마이드/소듐아크릴로일다이메틸타우레이트코폴리머 세틸알코올 토코페롤\n\n멜라B3 세럼\n정제수 다이메티콘 나이아신아마이드 글리세린 프로필렌글라이콜 폴리실리콘-11 실리카 비스-피이지/피피지-16/16피이지/피피지-16/16다이메티콘 레인보우랙추출물 2-머캅토니코티노일글라이신 피이지-20메틸글루코오스세스퀴스테아레이트 소듐하이알루로네이트 소듐하이드록사이드 소듐티오설페이트 카르노신 폴록사머338 암모늄폴리아크릴로일다이메틸타우레이트 다이포타슘글리시리제이트 카프릴릭/카프릭트라이글리세라이드 카프릴로일살리실릭애씨드 카프릴릴글라이콜 시트릭애씨드 트라이소듐에틸렌다이아민다이석시네이트 잔탄검 펜틸렌글라이콜 옥틸도데칸올 레티닐팔미테이트 토코페롤 펜타에리스리틸테트라-다이-t-부틸하이드록시하이드로신나메이트 페녹

### 1. Document - Custom

In [3]:
from langchain.schema import Document

doc = Document(
    page_content="Hello!",
    metadata={"source": "test"}
)

doc

Document(metadata={'source': 'test'}, page_content='Hello!')

In [9]:
import json
from langchain_core.documents import Document

# 변환할 JSONL 파일 경로를 지정하세요.
jsonl_file_path = file_path

documents = []

try:
    # 파일을 한 줄씩 읽어 메모리 문제를 방지합니다.
    with open(jsonl_file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            # 비어있는 줄은 건너뜁니다.
            if not line.strip():
                continue
            
            try:
                # 한 줄을 JSON 객체(파이썬 딕셔너리)로 변환합니다.
                obj = json.loads(line)

                # 1. page_content 생성 (핵심 텍스트 내용)
                page_content = obj.get('pdf_summary', '')
                
                # 2. metadata 생성 (부가 정보)
                # .get() 메소드를 사용하면 특정 키가 없어도 오류 없이 안전하게 값을 가져올 수 있습니다.
                metadata = {
                    'name': obj.get('product_name'),
                    'brand': obj.get('brand'),
                    'category': "->".join(obj.get('category_path')),
                    'price': obj.get('original_price'),
                    'volume': obj.get('용량'),
                    'manufacturer': obj.get('제조/판매업자'),
                    'review': obj.get('review_summary'),
                    'ingredients': obj.get('전성분') # 전성분도 메타데이터로 관리하여 필터링에 활용 가능
                }

                # 3. LangChain Document 객체 생성
                doc = Document(page_content=page_content, metadata=metadata)
                
                # 생성된 객체를 리스트에 추가
                documents.append(doc)

            except json.JSONDecodeError:
                print(f"경고: {i+1}번째 줄에서 JSON 파싱 오류가 발생했습니다. 해당 줄을 건너뜁니다.")
            except Exception as e:
                print(f"경고: {i+1}번째 줄 처리 중 오류 발생: {e}")

    # 변환 결과 확인
    print(f"성공적으로 총 {len(documents)}개의 Document 객체를 생성했습니다.")
    
    # 첫 번째 Document 객체가 어떻게 만들어졌는지 샘플로 확인
    if documents:
        print("\n--- 첫 번째 Document 객체 샘플 ---")
        print(documents[0])
        
        print("\n--- 첫 번째 Document의 메타데이터 ---")
        print(documents[0].metadata)


except FileNotFoundError:
    print(f"오류: 파일을 찾을 수 없습니다 - {jsonl_file_path}")
except Exception as e:
    print(f"알 수 없는 오류가 발생했습니다: {e}")

경고: 4번째 줄 처리 중 오류 발생: 1 validation error for Document
page_content
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
경고: 16번째 줄 처리 중 오류 발생: 1 validation error for Document
page_content
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
경고: 21번째 줄 처리 중 오류 발생: 1 validation error for Document
page_content
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
경고: 42번째 줄 처리 중 오류 발생: 1 validation error for Document
page_content
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
경고: 77번째 줄 처리 중 오류 발생: 1 validation error

In [7]:
import json
from langchain_core.documents import Document

documents = []

try:
	with open(file_path, 'r', encoding='utf-8') as f:
		for i, line in enumerate(f):
			if not line.strip():
				continue
			
			try:
				obj = json.loads(line)
				
				# 1. page_content 생성
				page_content = obj.get('pdf_summary') or ''
					# 키가 없거나 값이 null일 때 모두 None 반환
					# None or '' : 앞의 값이 거짓이므로 뒤의 값 선택
				
				# 2. metadata 생성
				metadata = {
					'name': obj.get('product_name'),
          'brand': obj.get('brand'),
          # Chroma에는 list가 들어갈 수 없기때문에 변형됨
          # 추후 Pinecone에서는 리스트 그대로 넣어서 진행할 것
          'category': "->".join(obj.get('category_path')),
          'price': obj.get('original_price'),
          'volume': obj.get('용량'),
          'manufacturer': obj.get('제조/판매업자'),
          'review': obj.get('review_summary'),
          'ingredients': obj.get('전성분')
				}
				
				# 3. Langchain Document 객체 생성
				doc = Document(page_content= page_content, metadata=metadata)
				# 생성된 객체를 리스트에 추가
				documents.append(doc)
			except json.JSONDecodeError:
				print(f"경고: {i+1}번째 줄에서 JSON 파싱 오류가 발생했습니다. 해당 줄을 건너뜁니다.")
			except Exception as e:
				print(f"경고: {i+1}번째 줄 처리 중 오류 발생: {e}")
	
	# 결과 확인
	print(f"성공적으로 총 {len( documents)}개의 Document 객체를 생성했습니다.")
	
	# 샘플 확인
	if documents:
		print("\n--- 첫 번째 Document 객체 샘플 ---")
		print(documents[0].page_content)
		
		print("\n--- 첫 번째 Document의 메타데이터 ---")
		print(documents[0].metadata)

except FileNotFoundError:
	print(f"오류: 파일을 찾을 수 없습니다. - {file_path}")
except Exception as e:
	print(f"알 수 없는 오류가 발생했습니다: {e}")

성공적으로 총 900개의 Document 객체를 생성했습니다.

--- 첫 번째 Document 객체 샘플 ---
# 고객 중심 컨셉 분석 리포트: 라로슈포제 시카플라스트 멀티 리페어 크림

---

## 1. 제품 기본 정보 (Product Identity)

*   **제품명**: 시카플라스트 멀티 리페어 크림 (Cicaplast Multi Repair Cream)
*   **브랜드**: 라로슈포제 (LA ROCHE POSAY)
*   **핵심 효능**: 피부 진정, 수분 공급, 탄력 개선, 피부 장벽 강화

---

## 2. 컨셉 심층 분석 (In-depth Concept Analysis)

*   **타겟 고객 및 문제 제기 (Problem)**:
    *   **타겟**: 민감성 피부를 가진 모든 연령대의 소비자, 특히 외부 자극으로 인해 피부 장벽이 약해지고 건조함, 탄력 저하, 피부결 개선이 필요한 고객. 한국인 피부에 대한 맞춤 솔루션을 찾는 고객.
    *   **고민**:
        *   환절기, 계절 변화, 미세먼지 등 외부 자극에 쉽게 민감해지고 붉어지는 피부.
        *   속건조로 인한 당김, 푸석함, 피부결이 거칠어지는 문제.
        *   탄력 저하로 느껴지는 피부 처짐 및 늘어난 모공.
        *   손상된 피부 장벽으로 인한 전반적인 피부 건강 악화.

*   **핵심 컨셉 및 해결책 (Solution)**:
    *   **한 줄 정의**: 외부 자극으로부터 피부를 보호하고 속부터 탄탄하게 채워주는 더모 코스메틱 솔루션.
    *   **해결 방식**:
        *   **이중 케어**: 피부 겉을 보호막처럼 감싸 외부 자극을 차단하고, 피부 속 깊숙이 유효 성분을 전달하여 수분과 영양을 공급합니다.
        *   **고효능 성분 배합**: 판테놀, 마데카소사이드 등 진정 및 장벽 강화 성분과 트라이바이오마™ 콤플렉스를 통해 피부 근본적인 힘을 길러 건강한 피부 환경을 조성합니다

### 2. Chunk Splitter

In [10]:
# Cell 8: 텍스트 분할 테스트
"""
텍스트를 어떻게 나눌지 실험해보자
"""

from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 생성
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # 각 청크의 최대 크기
    chunk_overlap=100,  # 청크 간 겹치는 부분
    separators=["\n\n", "\n", ". ", " "]  # 분할 우선순위
)

# 첫 번째 문서로 테스트
test_content = documents[0].page_content
chunks = text_splitter.split_text(test_content)

print(f"원본 문서: {len(test_content)} 글자")
print(f"분할 결과: {len(chunks)}개 청크\n")

# 처음 2개 청크 확인
for i, chunk in enumerate(chunks[:2]):
    print(f"청크 {i+1}:")
    print(chunk)
    print("-" * 30)

원본 문서: 5137 글자
분할 결과: 16개 청크

청크 1:
# 고객 중심 컨셉 분석 리포트: 라로슈포제 시카플라스트 멀티 리페어 크림

---

## 1. 제품 기본 정보 (Product Identity)

*   **제품명**: 시카플라스트 멀티 리페어 크림 (Cicaplast Multi Repair Cream)
*   **브랜드**: 라로슈포제 (LA ROCHE POSAY)
*   **핵심 효능**: 피부 진정, 수분 공급, 탄력 개선, 피부 장벽 강화

---

## 2. 컨셉 심층 분석 (In-depth Concept Analysis)
------------------------------
청크 2:
---

## 2. 컨셉 심층 분석 (In-depth Concept Analysis)

*   **타겟 고객 및 문제 제기 (Problem)**:
    *   **타겟**: 민감성 피부를 가진 모든 연령대의 소비자, 특히 외부 자극으로 인해 피부 장벽이 약해지고 건조함, 탄력 저하, 피부결 개선이 필요한 고객. 한국인 피부에 대한 맞춤 솔루션을 찾는 고객.
    *   **고민**:
        *   환절기, 계절 변화, 미세먼지 등 외부 자극에 쉽게 민감해지고 붉어지는 피부.
        *   속건조로 인한 당김, 푸석함, 피부결이 거칠어지는 문제.
        *   탄력 저하로 느껴지는 피부 처짐 및 늘어난 모공.
        *   손상된 피부 장벽으로 인한 전반적인 피부 건강 악화.
------------------------------


In [8]:
len(documents)

900

### 3. Embedding & Vector Store

##### Error : Document가 너무 길어서 처리 임베딩에서 처리 불가 - 청킹 필수

In [ ]:
# from langchain_openai import OpenAIEmbeddings
# from langchain_chroma import Chroma

# # 임베딩 준비
# embeddings = OpenAIEmbeddings()

# # 벡터스토어 생성
# vectorstore = Chroma.from_documents(
#     documents=documents,
#     embedding=embeddings
# )

BadRequestError: Error code: 400 - {'error': {'message': 'Requested 3478444 tokens, max 300000 tokens per request', 'type': 'max_tokens_per_request', 'param': None, 'code': 'max_tokens_per_request'}}

# 다시 오류로 인한 재시도

In [18]:
from pathlib import Path
from langchain.storage import LocalFileStore, create_kv_docstore

path = Path.cwd() / "data"
print(path)

# 부모 문서를 위한 Document Store (메모리 내 저장소 사용)
store = LocalFileStore(path)
docstore = create_kv_docstore(store)

c:\cursor\langgraph_baseline\oliveyoung\data


In [19]:
# Cell 9: ParentDocumentRetriever 소개
"""
## Step 5: ParentDocumentRetriever 사용하기

우리가 원하는 것:
1. 작은 청크로 검색 (정확도 ↑)
2. 전체 문서 반환 (맥락 유지)

ParentDocumentRetriever가 정확히 이걸 해준다!
"""

from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

print("🎯 ParentDocumentRetriever의 작동 방식:")
print("1. Parent 문서를 작은 Child 청크로 분할")
print("2. Child 청크를 벡터DB에 저장 (검색용)")
print("3. Parent 문서는 별도 저장소에 보관")
print("4. 검색 시: Child 검색 → Parent 반환")

# Cell 10: ParentDocumentRetriever 설정
"""
이제 실제로 설정해보자!
"""

# 1. 임베딩 모델 준비
embeddings = OpenAIEmbeddings()

# 2. 벡터스토어 준비 (Child 청크 저장용)
persist_directory = "./chroma_db"  # 데이터를 저장할 폴더 이름

vectorstore = Chroma(
    collection_name="cosmetic_chunks",
    embedding_function=embeddings,
    persist_directory=persist_directory
)

# 3. Parent 문서 저장소
# docstore = InMemoryStore()

# 4. ParentDocumentRetriever 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=text_splitter,  # 아까 만든 분할기 사용
)

print("✅ ParentDocumentRetriever 설정 완료!")
print(f"  - Child 청크 크기: {text_splitter._chunk_size}")
print(f"  - 청크 겹침: {text_splitter._chunk_overlap}")

🎯 ParentDocumentRetriever의 작동 방식:
1. Parent 문서를 작은 Child 청크로 분할
2. Child 청크를 벡터DB에 저장 (검색용)
3. Parent 문서는 별도 저장소에 보관
4. 검색 시: Child 검색 → Parent 반환
✅ ParentDocumentRetriever 설정 완료!
  - Child 청크 크기: 500
  - 청크 겹침: 100


In [13]:
# # 2. (저장 후 계속 사용) 저장된 벡터 스토어 불러오기
# # 프로그램이 재시작되거나, 다른 파일에서 사용할 때
# print("저장된 벡터 스토어를 불러옵니다...")
# vectorstore_loaded = Chroma(
#     persist_directory=persist_directory,
#     embedding_function=embedding_function
# )

In [8]:
# Cell 11: 문서 추가하기
"""
이제 우리 문서들을 추가해보자
"""

print("📥 문서 추가 중...")

# 문서 추가 - ParentDocumentRetriever가 자동으로 처리!
parent_retriever.add_documents(documents)

print("\n✅ 문서 추가 완료!")

# 저장된 내용 확인
print(f"\n📊 저장 통계:")
print(f"  - Parent 문서 수: {len(list(docstore.yield_keys()))}")
print(f"  - Child 청크 수: {vectorstore._collection.count()}")

# Parent 문서 ID 확인
print("\n저장된 Parent 문서 ID:")
for key in list(docstore.yield_keys())[:5]:  # 처음 5개만
    print(f"  - {key}")

📥 문서 추가 중...


InternalError: ValueError: Batch size of 12793 is greater than max batch size of 5461

In [20]:
# 한 번에 처리할 문서의 수를 정합니다 (서버나 문서 크기에 따라 조절).
# 예를 들어 50개 또는 100개로 시작해 보세요.
doc_batch_size = 100 

# 전체 문서를 정해진 배치 크기만큼 잘라 루프를 실행합니다.
for i in range(0, len(documents), doc_batch_size):
    # documents 리스트에서 일부만 잘라 batch_docs를 만듭니다.
    batch_docs = documents[i : i + doc_batch_size]
    
    # 잘라낸 작은 묶음(batch)만 retriever에 추가합니다.
    parent_retriever.add_documents(batch_docs)
    
    # 진행 상황을 확인하기 위한 로그 (선택 사항)
    print(f"문서 {i + len(batch_docs)} / {len(documents)} 처리 완료")

print("모든 문서 추가 완료!")

# 저장된 내용 확인
print(f"\n📊 저장 통계:")
print(f"  - Parent 문서 수: {len(list(docstore.yield_keys()))}")
print(f"  - Child 청크 수: {vectorstore._collection.count()}")

# Parent 문서 ID 확인
print("\n저장된 Parent 문서 ID:")
for key in list(docstore.yield_keys())[:5]:  # 처음 5개만
    print(f"  - {key}")

문서 100 / 841 처리 완료
문서 200 / 841 처리 완료
문서 300 / 841 처리 완료
문서 400 / 841 처리 완료
문서 500 / 841 처리 완료
문서 600 / 841 처리 완료
문서 700 / 841 처리 완료
문서 800 / 841 처리 완료
문서 841 / 841 처리 완료
모든 문서 추가 완료!

📊 저장 통계:
  - Parent 문서 수: 841
  - Child 청크 수: 14429

저장된 Parent 문서 ID:
  - 00443851-0489-4765-ab87-8feefc19656a
  - 00771e26-810f-46d0-ad43-84e384b6f7ec
  - 00a5005a-387e-4cd7-80c1-0c159cb69cc9
  - 00b4a0be-fde3-4c2f-96f0-30f03022b8f2
  - 00c36a70-fe49-4dea-8e90-a39d27b79631


In [13]:
# Cell 12: 첫 검색 테스트
"""
## Step 6: 검색해보기!

드디어 검색을 해보자
"""

# 검색 쿼리
query = """
## 컨셉 A

1. **제품 컨셉 (핵심 스토리)**
    - "**들키지마! 또비니의 모범생 쌩얼 치트키 톤업 선쿠션**"은 인플루언서 또비니의 시그니처 '꾸안꾸' 철학을 담아, 학교 규제를 영리하게 피하고 싶어 하는 학생 타겟에게 본연의 피부인 듯한 자연스러운 톤업과 자외선 차단 효과를 동시에 제공하는 '믿고 쓰는' 피부 표현 치트키입니다. 마치 '원래 좋았던 피부'처럼 연출해주는 간편하고 확실한 솔루션을 선사합니다.
2. **맞춤형 컨셉 설명**
    - 이 제품은 또비니 채널의 가장 강력한 페인 포인트인 '학교 화장 규제'를 정면으로 돌파하는 맞춤형 제품입니다. 또비니가 '연한 메이크업', '학생 메이크업' 영상에서 강조하는 '들키지 않는 자연스러움'과 '피부 본연의 건강한 느낌'을 쿠션 하나로 구현합니다. 팬덤이 간절히 원하는 '과하지 않으면서도 예뻐지는' 방법을 또비니의 이름으로 직접 제시하여, 단순히 제품을 넘어 '똑똑한 학교생활'의 필수템이자 '자신감 향상'의 도구로 포지셔닝합니다. 또비니 특유의 친근하고 솔직한 톤앤매너('들키지 마', '찐템')가 제품 컨셉과 마케팅 언어에 그대로 반영되어 팬덤에게 압도적인 신뢰와 공감을 불러일으킬 것입니다.
3. **선정 근거**
    - **데이터 근거 1: [롱폼/숏폼] 성공 영상 속 '학교 메이크업'의 폭발적 반응 및 규제 언급**
        - **영상 1 ("Latte 중딩들의 화장법"):** 숏폼 영상으로, "선크림도 색 있으면 안 된다", "벌점", "클렌징 티슈로 닦인다" 등 학교 규제에 대한 댓글(수천 개)이 압도적으로 많았으며, '들키지 않는 자연스러운 톤업'에 대한 강력한 미충족 니즈가 확인되었습니다. 또비니 또한 '티 안 나는 화장법'을 강조하며 학생 시청자들의 공감을 샀습니다.
        - **영상 49 ("화장 안 했는데요? 헐 수 있는 진~짜 자연스러운 학생 메이크업"):** 또비니가 선쿠션을 사용하며 "요렇게만 해줘도 피부가 조금은 밝아 보인다고", "진짜 자연스러운 톤업 선쿠션"이라 언급하고, 팬덤은 '학교 화장 들키지 않는 법', '선생님께 안 걸리는 법' 등에 대한 질문이 쏟아졌습니다. (최근 6개월 데이터)
    - **데이터 근거 2: 반복적으로 나타난 구독자 니즈 '피부 고민 해결' 및 '가성비'**
        - 팬덤의 화장품 고민 중 '여드름', '트러블 흔적', '홍조', '예민 피부' 등 피부 트러블 관련 니즈가 높게 나타났습니다. (영역 2.2, 2.5) 이는 단순한 톤업을 넘어 피부 부담이 적고 진정 효과까지 있는 선제품에 대한 잠재적 수요를 시사합니다.
        - '학생이라 돈이 없어서', '용돈 부족' 등 경제적 제약으로 인해 '가성비 좋은' 합리적인 가격대의 제품에 대한 요구가 높습니다. (영역 2.2, 2.6) 선쿠션은 드럭스토어에서 쉽게 접할 수 있는 카테고리로, 합리적인 가격으로 출시하기에 용이합니다.
    - **데이터 근거 3: 또비니 채널의 '찐템' 신뢰도와 '실용성' 가치관과의 시너지**
        - 또비니는 자신의 '찐템', '필수템' 추천에 대한 팬덤의 압도적인 신뢰를 받고 있으며("언니 픽은 믿고 사지", "바로 주문했어요" - 영역 2.3). 또비니의 '꾸안꾸' 뷰티 철학과 '실용성' 중심의 제품 선택 기준(영역 1.3)이 '데일리 톤업 선쿠션'과 완벽하게 일치합니다. 이는 제품 출시 시 폭발적인 구매 동기로 작용할 것입니다.
4. **제품 스펙 (컨셉 구현 방안)**
    - **카테고리:** 톤업 선쿠션 (SPF 50+ PA++++ / 미백, 주름 개선, 자외선 차단 3중 기능성)
    - **타겟 페르소나:** '학교 규제 속에서도 예뻐 보이고 싶은' 중고등학생 및 '자연스러운 쌩얼 메이크업'을 선호하는 20대 초반 여성.
    - **핵심 특성:**
        - **'들키지 않는' 자연스러운 톤업:** 백탁 없이 피부에 얇게 밀착되어 원래 좋은 피부인 것처럼 자연스럽게 톤업되는 미세한 핑크빛/살구빛 베이지 톤업 (쿨톤/웜톤 모두 사용 가능).
        - **피부 진정 및 보호:** 티트리, 병풀 추출물 등 피부 트러블 진정 및 민감성 피부에 부담 없는 순한 성분 함유. 여드름성 피부 사용 적합 테스트 완료.
        - **끈적임 없는 보송한 마무리:** 피지 컨트롤 파우더를 함유하여 답답함 없이 끈적이지 않고 뽀송하게 마무리되어, 마스크나 옷에 묻어남 최소화.
        - **강력한 지속력 & 밀착력:** 땀과 유분에도 쉽게 지워지지 않고 하루 종일 무너짐 없는 롱래스팅 포뮬러로, '닦이면 그만'이라는 불만 해소.
        - **간편한 휴대성:** 한 손에 쏙 들어오는 슬림하고 튼튼한 쿠션 용기로, 언제 어디서든 쉽고 빠르게 덧바를 수 있도록 디자인.

---

## 컨셉 B

1. **제품 컨셉 (핵심 스토리)**
    - "**칼퇴 후 완벽 변신! 또비니의 갓생 퀵 터치 멀티 팔레트**"는 또비니의 '갓생 (God-life)' 모토처럼 바쁜 일상 속에서도 놓칠 수 없는 자기관리를 위한, 빠르고 효율적인 '원샷 올킬' 메이크업 솔루션입니다. 눈과 볼에 생기를 더하고 피부 결점을 커버하며 '생기로운 무결점 동안 광채'를 연출해주는 멀티 유즈 팔레트로, 마치 또비니의 찐템처럼 '믿고 쓰는' 만능 뷰티 아이템입니다.
2. **맞춤형 컨셉 설명**
    - 이 제품은 또비니가 '칼퇴 후 완벽 변신', '5분 컷 메이크업' 등 바쁜 직장인/학생들을 위한 실용적이고 빠른 메이크업 팁을 제공하는 영상들과 완벽하게 부합합니다. 또비니의 '과즙블러', '찰떡 피부결', '반전매력광'이라는 뷰티 페르소나 키워드들을 멀티 팔레트 하나에 담아, 팬덤이 동경하는 그녀의 '생기 있는 동안' 이미지를 쉽고 빠르게 따라 할 수 있도록 돕습니다. 팬덤이 겪는 '화장품 정보 부족', '간편한 루틴 갈증'을 해소해주고, '합리적인 가격대의 고성능 제품'이라는 니즈를 충족하는 '또비니 픽' 찐템으로 자리매김할 것입니다.
3. **선정 근거**
    - **데이터 근거 1: [롱폼/숏폼] '퀵 메이크업' 및 '멀티 유즈' 콘텐츠의 팬덤 호응**
        - **영상 5 ("상견례 프리패스상 메이크업"):** "애교의 핑크를 살짝 추가요", "촉촉해지고 얇게 광을 내실 수가 있어" 등 블러셔와 애교살, 촉촉한 베이스 표현이 강조되며, 팬덤이 '동안'과 '생기'를 위한 핵심 포인트를 또비니에게서 배우고자 함을 보여줍니다.
        - **영상 51 ("1차 세안으로 끝! 파데+블러셔+자차+립까지 싹-원샷 올킬🔥"):** '올킬'이라는 표현에서 팬덤의 '간편하고 효율적인' 루틴에 대한 강력한 니즈가 드러납니다. 또비니의 '간편함'에 대한 가치관과도 일치합니다. (최근 6개월 데이터)
    - **데이터 근거 2: 팬덤의 '동안/생기 있는 인상' 워너비 및 '제품 정보' 갈증**
        - 팬덤은 또비니의 '귀엽고 사랑스러운 동안 이미지'와 '과즙상' 메이크업('토마토 블러셔', '인간 복숭아' - 영역 2.1)에 높은 관심을 보입니다. 이는 핑크/피치/코랄 계열의 블러셔와 눈매를 살리는 컬러 조합에 대한 수요로 이어집니다.
        - '어디꺼예요?', '제품명 알려주세요' 등 사용된 제품의 정확한 정보 요구가 압도적입니다. (영역 2.1) 멀티 팔레트는 여러 컬러와 기능을 한 번에 제공하여 이러한 정보 탐색의 어려움을 줄여줍니다.
    - **데이터 근거 3: 또비니의 '봄웜톤' 개인화 접근 및 '갓성비' 브랜딩 시너지**
        - 또비니는 본인의 '봄웜라' 퍼스널 컬러를 바탕으로 한 제품 추천이 많으며, 팬덤 또한 '봄웜라'에 맞는 제품 선택에 대한 고민을 공유합니다. (영역 1.4, 2.2) 특정 퍼스널 컬러에 최적화된 팔레트는 팬덤의 개인화 니즈를 충족할 수 있습니다.
        - 또비니의 '갓성비 찐템' 브랜딩 DNA(영역 1.4)와 '합리적인 가격대'를 선호하는 팬덤의 소비 성향(영역 2.6)을 고려할 때, 하나의 제품으로 여러 효과를 낼 수 있는 멀티 팔레트는 가격 대비 높은 효용성을 제공하여 구매를 유도할 수 있습니다.
4. **제품 스펙 (컨셉 구현 방안)**
    - **카테고리:** 멀티 유즈 크림/파우더 팔레트 (아이, 치크, 컨투어, 애교살 연출)
    - **타겟 페르소나:** '바쁜 일상 속에서도 생기 있고 완벽한 메이크업을 원하지만, 여러 제품 사용이 번거로운' 20대 초중반 직장인 및 대학생.
    - **핵심 특성:**
        - **'생기로운 무결점 동안 광채' 컬러 조합:** 또비니의 시그니처 '봄웜' 무드를 담은 핑크/피치/코랄 계열의 2-3가지 블러셔 컬러와, 피부 결점을 자연스럽게 커버하고 음영을 줄 수 있는 베이지/뮤트 브라운 계열의 2가지 쉐이드 (컨실러/쉐딩 겸용)로 구성.
        - **'퀵 터치' 멀티 유즈 포뮬러:**
            - **크림 블러셔/컨실러:** 손가락으로도 쉽게 블렌딩되는 부드러운 크림 제형으로, 빠르고 자연스러운 발색과 밀착력 제공.
            - **파우더/글리터:** 미세하고 고운 입자의 파우더 쉐이딩, 은은한 광채를 더하는 하이라이터/애교살 글리터.
        - **'찰떡 피부결' 표현:** 피부에 녹아들 듯 얇게 밀착되어 모공 및 요철을 자연스럽게 블러 처리하고, 건조함 없이 은은한 윤광을 선사.
        - **휴대성 & 실용성:** 얇고 가벼운 미니 팔레트 형태로, 파우치에 넣어 다니며 언제든 '퀵 리프레시' 가능. 거울 내장으로 편리함 극대화.
        - **초보자 친화적:** 또비니가 직접 알려주는 '퀵 터치' 활용법 미니 가이드북 또는 QR 코드 영상 제공으로 누구나 쉽게 '갓생 메이크업' 연출.

"""

query2 = "스킨케어"

# print(f"🔍 검색어: '{query}'")
print("-" * 50)

# 검색 실행
results = parent_retriever.invoke(query2)

print(f"\n검색 결과: {len(results)}개 제품")

# 결과 확인
for i, doc in enumerate(results[:5]):  # 상위 3개만
    print(f"\n[{i+1}] {doc.metadata['name']}")
    print(f"  브랜드: {doc.metadata['brand']}")
    print(f"  가격: {doc.metadata['price']:,}원")
    # print(f"  내용 길이: {len(doc.page_content)} 글자")  # 전체 문서!
    print(f"  내용: {(doc.page_content[:50])}")  # 전체 문서!
    print(f"  리뷰: {doc.metadata['review'][:50]}")

--------------------------------------------------


NameError: name 'parent_retriever' is not defined